<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.5_Praktikum_Image_Classification_dengan_CNN.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.5: Praktikum Image Classification End-to-End dengan CNN
**Tujuan Pembelajaran:**
1. Membangun pipeline pelatihan klasifikasi citra end-to-end lengkap dengan validasi metrik berkala.
2. Menerapkan regularisasi *weight decay*, optimasi AdamW, dan penjadwalan *Cosine Annealing*.
3. Menghitung dan memvisualisasikan Confusion Matrix multi-kelas untuk evaluasi akurasi sistem industri.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch Versi: {torch.__version__}")

### Bagian 1: Sintesis Dataset Multi-Kelas Mutu Kematangan TBS Sawit (3 Kelas)

In [ ]:
# 3 Kelas: 0: Mentah, 1: Matang, 2: Lewat Matang
def generate_harvest_dataset(num_samples=450):
    X = np.zeros((num_samples, 3, 32, 32), dtype=np.float32)
    y = np.zeros(num_samples, dtype=np.int64)
    
    for i in range(num_samples):
        cls = i % 3
        y[i] = cls
        if cls == 0:   # Mentah (Hitam Keunguan)
            r, g, b = 0.20, 0.18, 0.22
        elif cls == 1: # Matang Sempurna (Oranye Kemerahan)
            r, g, b = 0.85, 0.40, 0.10
        else:          # Lewat Matang (Merah Tua Kusam)
            r, g, b = 0.45, 0.15, 0.08
            
        noise = np.random.normal(0, 0.04, (3, 32, 32))
        X[i, 0, :, :] = r + noise[0]
        X[i, 1, :, :] = g + noise[1]
        X[i, 2, :, :] = b + noise[2]
        
    X = np.clip(X, 0.0, 1.0)
    return torch.tensor(X), torch.tensor(y)

X_data, y_data = generate_harvest_dataset(450)
ds_total = TensorDataset(X_data, y_data)
train_ds, val_ds = torch.utils.data.random_split(ds_total, [360, 90])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
print(f"Dataset siap: Train = {len(train_ds)}, Val = {len(val_ds)}")

### Bagian 2: Membangun Arsitektur dan Melatih Model End-to-End

In [ ]:
class IndustrialClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = IndustrialClassifier(num_classes=3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

epochs = 10
train_losses, val_accs = [], []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for bx, by in train_loader:
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * bx.size(0)
        
    scheduler.step()
    train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(train_loss)
    
    # Evaluasi
    model.eval()
    correct = 0
    with torch.no_grad():
        for vx, vy in val_loader:
            preds = model(vx).argmax(dim=1)
            correct += (preds == vy).sum().item()
    acc = correct / len(val_loader.dataset)
    val_accs.append(acc)
    print(f"Epoch [{epoch+1:02d}/{epochs}] - Loss: {train_loss:.4f} | Val Accuracy: {acc*100:.2f}%")

### Bagian 3: Komputasi Confusion Matrix dan Visualisasi

In [ ]:
# Evaluasi matriks kontingensi
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for vx, vy in val_loader:
        preds = model(vx).argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(vy.cpu().numpy())

num_classes = 3
conf_mat = np.zeros((num_classes, num_classes), dtype=int)
for true_l, pred_l in zip(all_labels, all_preds):
    conf_mat[true_l, pred_l] += 1

print("Confusion Matrix:")
print(conf_mat)

# Visualisasi Matriks
class_names = ['Mentah', 'Matang', 'Lewat Matang']
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(conf_mat, cmap='Blues')
plt.colorbar(im)

ax.set_xticks(np.arange(num_classes))
ax.set_yticks(np.arange(num_classes))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
plt.xlabel('Prediksi Model')
plt.ylabel('Label Ground Truth')
plt.title('Confusion Matrix Sortasi TBS Kelapa Sawit')

# Nilai teks pada sel
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, str(conf_mat[i, j]), ha='center', va='center',
                color='white' if conf_mat[i, j] > conf_mat.max()/2 else 'black', fontweight='bold')

plt.tight_layout()
plt.savefig('praktikum_confusion_matrix_12_5.png', dpi=200)
plt.close()
print("[OK] Confusion matrix disimpan sebagai 'praktikum_confusion_matrix_12_5.png'")